In [1]:
!rm -rf /kaggle/working/*

In [2]:
import os
import sys
import logging
import warnings
from PIL import Image

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*can only test a child process.*")
warnings.filterwarnings("ignore", message=".*Exception ignored in.*")
warnings.filterwarnings("ignore", message=".*_MultiProcessingDataLoaderIter.*")
warnings.filterwarnings("ignore", message=".*_shutdown_workers.*")

logging.disable(logging.CRITICAL)

os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

Image.MAX_IMAGE_PIXELS = None

class SuppressOutput:
    def write(self, x):
        pass

    def flush(self):
        pass

sys.stderr = SuppressOutput()

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image
import gradio as gr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchsummary import summary

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score,
    roc_curve, auc
)

from tqdm.notebook import tqdm
from tabulate import tabulate
from IPython.display import clear_output

%matplotlib inline

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Seed: {SEED}")
print(f"Device: {DEVICE}")

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

In [ ]:
DATA_ROOT      = "/kaggle/input/datasets/evilspirit05/ecg-analysis/ecg_data"
IMAGE_SIZE     = 224
BATCH_SIZE     = 4
EPOCHS         = 200
LEARNING_RATE  = 3e-5
RANDOM_SEED    = 42
patience = 20
NUM_CLASSES = 4

In [ ]:
image_paths = []

for class_name in os.listdir(DATA_ROOT):
    class_dir = os.path.join(DATA_ROOT, class_name)

    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                image_paths.append((os.path.join(class_dir, img_name), class_name))

# Randomly select 25 images
sample_images = random.sample(image_paths, 25)

# Plot 5x5 grid
fig, axes = plt.subplots(5, 5, figsize=(15, 15))

for ax, (img_path, label) in zip(axes.ravel(), sample_images):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((IMAGE_SIZE, IMAGE_SIZE))

    ax.imshow(img)
    ax.set_title(label.replace("_", " "), fontsize=8)
    ax.axis("off")

plt.suptitle("Random ECG Samples (5x5 Grid)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
class_counts = {}

for class_name in os.listdir(DATA_ROOT):
    class_path = os.path.join(DATA_ROOT, class_name)

    if os.path.isdir(class_path):
        class_counts[class_name] = len(os.listdir(class_path))

df = pd.DataFrame({"Class": class_counts.keys(),"Count": class_counts.values()})

plt.figure(figsize=(15, 6))

sns.barplot(data=df,x="Class",y="Count",palette="Set3")

plt.title("ECG Dataset Class Distribution", fontsize=16)
plt.xlabel("Classes", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.xticks(rotation=90)

for index, value in enumerate(df["Count"]):
    plt.text(index, value + 5, str(value), ha='center', fontsize=11)

plt.show()


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=None)

print(f"Classes : {full_dataset.classes}")
print(f"Total   : {len(full_dataset)} images")

In [ ]:
labels = [label for _, label in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.20,
    stratify=labels,
    random_state=RANDOM_SEED,
)

temp_labels = [labels[i] for i in temp_idx]

valid_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=RANDOM_SEED,
)

print(f"Train : {len(train_idx)} | Valid : {len(valid_idx)} | Test : {len(test_idx)}")

In [ ]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


train_subset = TransformSubset(Subset(full_dataset, train_idx), train_transforms)
valid_subset = TransformSubset(Subset(full_dataset, valid_idx), val_test_transforms)
test_subset  = TransformSubset(Subset(full_dataset, test_idx), val_test_transforms)


train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
imgs, labels = next(iter(train_loader))

print(f"Images shape: {imgs.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique labels in batch: {labels.unique().tolist()}")

In [ ]:
class_names = train_subset.subset.dataset.classes
print(class_names)

In [ ]:
class_to_idx = train_subset.subset.dataset.class_to_idx
print(class_to_idx)

In [ ]:


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 4

class MultiScaleBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.branch1 = nn.Conv2d(in_channels, out_channels // 4, kernel_size=1)
        self.branch2 = nn.Conv2d(in_channels, out_channels // 4, kernel_size=3, padding=1)
        self.branch3 = nn.Conv2d(in_channels, out_channels // 4, kernel_size=5, padding=2)
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1),
        )
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()
        self.bn = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(0.1)

    def forward(self, x):
        residual = self.residual(x)
        out = torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)
        out = self.bn(out)
        out = F.relu(out)
        out = self.dropout(out)
        out = out + residual
        return F.relu(out)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, in_channels, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.query = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.key = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.value = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.out_conv = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        q = self.query(x).view(batch, self.num_heads, self.head_dim, height * width).permute(0, 1, 3, 2)
        k = self.key(x).view(batch, self.num_heads, self.head_dim, height * width)
        v = self.value(x).view(batch, self.num_heads, self.head_dim, height * width).permute(0, 1, 3, 2)
        attn = (q @ k) * self.scale
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(batch, channels, height, width)
        out = self.out_conv(out)
        return x + out

class SimCardioNet(nn.Module):
    def __init__(self, num_classes=4, ssl_proj_dim=128, dropout=0.4):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.block1 = MultiScaleBlock(64, 256)
        self.block2 = MultiScaleBlock(256, 512)
        self.block3 = MultiScaleBlock(512, 1024)
        self.attention = MultiHeadSelfAttention(1024, num_heads=8)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.global_max_pool = nn.AdaptiveMaxPool2d((1, 1))
        self.fusion = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(1024, num_classes)
        self.projection_head = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, ssl_proj_dim),
        )

    def forward(self, x, ssl=False):
        x = self.initial(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.attention(x)
        avg_pool = self.global_avg_pool(x).view(x.size(0), -1)
        max_pool = self.global_max_pool(x).view(x.size(0), -1)
        x = torch.cat([avg_pool, max_pool], dim=1)
        x = self.fusion(x)
        if ssl:
            return F.normalize(self.projection_head(x), dim=1)
        return self.classifier(x)

model = SimCardioNet(num_classes=NUM_CLASSES).to(DEVICE)


summary(model, input_size=(3, 224, 224), batch_size=32, device=DEVICE.type)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
save_path = "best_ecg_model.pth"
best_val_loss = float("inf")
counter = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
epoch_results = []

print(f"Training on {DEVICE}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    train_bar = tqdm(train_loader, desc=f"Train Epoch {epoch}/{EPOCHS}", leave=False, unit="batch")
    for imgs, labels in train_bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_correct += (logits.argmax(1) == labels).sum().item()
        t_total += imgs.size(0)
        train_bar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = t_loss / t_total
    train_acc = t_correct / t_total * 100

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    val_bar = tqdm(valid_loader, desc=f"Valid Epoch {epoch}/{EPOCHS}", leave=False, unit="batch")
    with torch.no_grad():
        for imgs, labels in val_bar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, labels)
            v_loss += loss.item() * imgs.size(0)
            v_correct += (logits.argmax(1) == labels).sum().item()
            v_total += imgs.size(0)
            val_bar.set_postfix(loss=f"{loss.item():.4f}")
    val_loss = v_loss / v_total
    val_acc = v_correct / v_total * 100

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    epoch_results.append([epoch, train_loss, train_acc, val_loss, val_acc])

    clear_output(wait=True)
    print(f"Training on {DEVICE}\n")
    print(tabulate(
        epoch_results,
        headers=["Epoch", "Train Loss", "Train Accuracy (%)", "Val Loss", "Val Accuracy (%)"],
        tablefmt="heavy_grid",
        floatfmt=".4f",
        numalign="center",
        stralign="center"
    ))
    print()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }, save_path)
        print(f"Saved best model at epoch {epoch}\n")
    else:
        counter += 1
        print(f"No improvement ({counter}/{patience})\n")
        if counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print("\n=== Training Complete ===\n")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ECG Classifier — Training History", fontsize=15, fontweight="bold", y=1.02)

# ── Loss ─────────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs, history["train_loss"], "o-",  color="#4C72B0", linewidth=2, markersize=5, label="Train Loss")
ax.plot(epochs, history["val_loss"],   "s--", color="#DD8452", linewidth=2, markersize=5, label="Val Loss")
ax.set_title("Loss", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs, history["train_acc"], "o-",  color="#4C72B0", linewidth=2, markersize=5, label="Train Acc")
ax.plot(epochs, history["val_acc"],   "s--", color="#DD8452", linewidth=2, markersize=5, label="Val Acc")
ax.set_title("Accuracy", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
MODEL_PATH = "/kaggle/working/best_ecg_model.pth"

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model = SimCardioNet(num_classes=4).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded successfully!")
print("Epoch:", checkpoint["epoch"])
print("Val Loss:", checkpoint["val_loss"])
print("Val Acc:", checkpoint["val_acc"])

In [ ]:
class_names = ['abnormal_heartbeat_ecg_images','myocardial_infarction_ecg_images','normal_ecg_images','post_mi_history_ecg_images']

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",xticklabels=class_names,yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(all_labels, all_preds,target_names=class_names))

In [ ]:
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="ROC collecting"):
        images = images.to(DEVICE)
        outputs = model(images)

        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

n_classes = 4

all_labels_bin = label_binarize(all_labels, classes=[0,1,2,3])


plt.figure(figsize=(10,8))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC = {roc_auc:.3f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Per-Class ROC Curve (ECG Model)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))

for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(all_labels_bin[:, i], all_probs[:, i])
    ap_score = average_precision_score(all_labels_bin[:, i], all_probs[:, i])

    plt.plot(recall, precision, label=f"{class_names[i]} (AP = {ap_score:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Multiclass ECG)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
model.eval()

y_true, y_pred, y_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        y_true.append(labels.cpu())
        y_pred.append(preds.cpu())
        y_probs.append(probs.cpu())

y_true = torch.cat(y_true)
y_pred = torch.cat(y_pred)
y_probs = torch.cat(y_probs)

num_classes = 4

accuracy = (y_pred == y_true).float().mean().item()

precision_list, recall_list, f1_list = [], [], []

for c in range(num_classes):
    tp = ((y_pred == c) & (y_true == c)).sum().item()
    fp = ((y_pred == c) & (y_true != c)).sum().item()
    fn = ((y_pred != c) & (y_true == c)).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)

precision = sum(precision_list) / num_classes
recall = sum(recall_list) / num_classes
f1 = sum(f1_list) / num_classes

roc_auc_list = []

for c in range(num_classes):
    scores = y_probs[:, c].numpy()
    targets = (y_true == c).numpy()

    order = scores.argsort()[::-1]
    t = targets[order]
    s = scores[order]

    tp = t.cumsum()
    fp = (1 - t).cumsum()

    tpr = tp / (tp[-1] + 1e-8)
    fpr = fp / (fp[-1] + 1e-8)

    auc = torch.trapz(torch.tensor(tpr), torch.tensor(fpr)).item()
    roc_auc_list.append(auc)

roc_auc = sum(roc_auc_list) / num_classes

print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1 Score   : {f1:.4f}")
print(f"ROC AUC    : {roc_auc:.4f}")

In [ ]:
model.eval()

class_names = [
    'abnormal_heartbeat_ecg_images',
    'myocardial_infarction_ecg_images',
    'normal_ecg_images',
    'post_mi_history_ecg_images'
]

images, true_labels = next(iter(test_loader))

images = images.to(DEVICE)

with torch.no_grad():
    outputs = model(images)
    probs = torch.softmax(outputs, dim=1)
    pred_labels = torch.argmax(probs, dim=1)
    confidences = torch.max(probs, dim=1).values

images = images.cpu().permute(0, 2, 3, 1).numpy()
true_labels = true_labels.numpy()
pred_labels = pred_labels.cpu().numpy()
confidences = confidences.cpu().numpy()

plt.figure(figsize=(10, 6))

for i in range(len(images)):  
    plt.subplot(2, 2, i + 1)

    img = images[i]
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    plt.imshow(img)

    true_name = class_names[true_labels[i]]
    pred_name = class_names[pred_labels[i]]
    conf = confidences[i]

    color = "green" if true_labels[i] == pred_labels[i] else "red"

    plt.title(
        f"T: {true_name}\nP: {pred_name}\nC: {conf:.2f}",
        color=color,
        fontsize=9
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

correct = np.sum(y_true == y_pred)
wrong = np.sum(y_true != y_pred)
total = len(y_true)

correct_pct = (correct / total) * 100
wrong_pct = (wrong / total) * 100

labels = ["Correct", "Wrong"]
values = [correct, wrong]
percentages = [correct_pct, wrong_pct]

plt.figure(figsize=(8, 5))

ax = sns.barplot(x=labels, y=values, palette="Set1")

for i, v in enumerate(values):
    plt.text(i, v + 0.1, f"{v} ({percentages[i]:.2f}%)",
             ha="center", fontweight="bold")

plt.title(f"Full Dataset Prediction Results (Samples = {total})")
plt.ylabel("Count")

plt.show()

print(f"Total Samples: {total}")
print(f"Correct Predictions: {correct}")
print(f"Wrong Predictions: {wrong}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)

class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(15, 6))
sns.barplot(x=class_names, y=class_acc, palette="hsv")

plt.title("Per-Class Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Classes")

plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.show()

In [ ]:
model.eval()

all_confidences = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        confidences = torch.max(probs, dim=1).values

        all_confidences.extend(confidences.cpu().numpy())

all_confidences = np.array(all_confidences)

plt.figure(figsize=(10, 5))
sns.histplot(all_confidences, bins=20, kde=True, color="purple")

plt.title("Prediction Confidence Distribution (PyTorch)")
plt.xlabel("Confidence")
plt.ylabel("Count")
plt.show()

In [ ]:
model.eval()

y_true = []
y_pred = []
confidences = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(probs, dim=1)
        conf = torch.max(probs, dim=1).values

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        confidences.extend(conf.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
confidences = np.array(confidences)

df = pd.DataFrame({
    "confidence": confidences,
    "correct": y_true == y_pred
})

plt.figure(figsize=(15, 5))
sns.boxplot(x="correct", y="confidence", data=df, palette="Set2")

plt.xticks([0, 1], ["Wrong", "Correct"])
plt.title("Confidence vs Correctness")

plt.show()

In [ ]:
error_cm = cm.copy()
np.fill_diagonal(error_cm, 0)

plt.figure(figsize=(12, 8))

sns.heatmap(
    error_cm,
    annot=True,
    fmt="d",
    cmap="Reds",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Most Confused Class Pairs")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.tight_layout()
plt.show()

In [ ]:

last_conv_name = None
last_conv_layer = None

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        last_conv_name = name
        last_conv_layer = module

print(f"Last Conv2D Layer: {last_conv_name}")
print(last_conv_layer)

print()
print("=" * 80)
print("ALL MODEL MODULES")
print("=" * 80)

for i, (name, module) in enumerate(model.named_modules()):
    print(f"[{i:3d}] {name:<50} {type(module).__name__}")

In [ ]:
target_layer = model.attention.out_conv

activations = None
gradients = None

def forward_hook(module, input, output):
    global activations
    activations = output.detach()

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0].detach()

forward_handle = target_layer.register_forward_hook(forward_hook)
backward_handle = target_layer.register_full_backward_hook(backward_hook)

print("Grad-CAM hooks registered successfully")
print(f"Target Layer: {target_layer}")

In [ ]:
images, labels = next(iter(test_loader))
images = images.to(DEVICE)

model.eval()

outputs = model(images)

print("Model output shape:", outputs.shape)

class_idx = outputs.argmax(dim=1)[0]

model.zero_grad()
outputs[0, class_idx].backward()

print("Activations shape:", activations.shape)
print("Gradients shape :", gradients.shape)

In [ ]:
def compute_gradcam(model, image_tensor, target_class=None):
    global activations, gradients

    model.eval()

    output = model(image_tensor)

    if target_class is None:
        target_class = output.argmax(dim=1).item()

    model.zero_grad()
    output[:, target_class].backward()

    pooled_grads = torch.mean(gradients, dim=(0, 2, 3))

    acts = activations[0]

    for i in range(acts.shape[0]):
        acts[i] *= pooled_grads[i]

    heatmap = torch.mean(acts, dim=0)
    heatmap = F.relu(heatmap)

    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    heatmap = heatmap.cpu().numpy()

    confidence = torch.softmax(output, dim=1).max().item()

    return heatmap, target_class, confidence


def overlay_heatmap(img_orig_uint8, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    h, w = img_orig_uint8.shape[:2]

    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)

    heatmap_color = cv2.applyColorMap(heatmap_uint8, colormap)
    heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    superimposed = cv2.addWeighted(
        img_orig_uint8,
        1 - alpha,
        heatmap_rgb,
        alpha,
        0
    )

    return superimposed, heatmap_rgb


images, labels = next(iter(test_loader))

idx = 0

img_tensor = images[idx:idx+1].to(DEVICE)
true_label = labels[idx].item()

img = images[idx].permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

img_uint8 = (img * 255).astype(np.uint8)

print(f"img_input shape : {tuple(img_tensor.shape)}")
print(f"true_label      : {true_label} → {class_names[true_label]}")

In [ ]:
heatmap, pred_label, confidence = compute_gradcam(model, img_tensor)

overlay_img, heatmap_rgb = overlay_heatmap(img_uint8, heatmap)

print(f"pred_label : {pred_label} → {class_names[pred_label]}")
print(f"confidence : {confidence:.4f}")

In [ ]:
n = 4

fig, axes = plt.subplots(n, 3, figsize=(14, n * 4))
col_titles = ["Original", "Heatmap Only", "Grad-CAM Overlay"]

model.eval()

images, labels = next(iter(test_loader))

for idx in range(n):
    img_tensor = images[idx:idx+1].to(DEVICE)
    true_label = labels[idx].item()

    img = images[idx].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_uint8 = (img * 255).astype(np.uint8)

    heatmap, pred_label, confidence = compute_gradcam(model, img_tensor)

    overlay_img, heatmap_rgb = overlay_heatmap(img_uint8, heatmap)

    images_list = [img_uint8, heatmap_rgb, overlay_img]

    color = "green" if true_label == pred_label else "red"

    for col, (ax, im) in enumerate(zip(axes[idx], images_list)):
        ax.imshow(im)
        ax.axis("off")

        if idx == 0:
            ax.set_title(col_titles[col], fontsize=13, fontweight="bold", pad=8)

        if col == 2:
            ax.set_title(
                f"True: {class_names[true_label]} | Pred: {class_names[pred_label]} | Conf: {confidence:.2f}",
                fontsize=10,
                color=color,
                fontweight="bold",
                pad=6
            )

plt.suptitle("Grad-CAM — 4 Test Samples", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def plot_gradcam_grid(test_loader, model, class_names, n=8, cols=4):
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    model.eval()
    count = 0

    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)

        for i in range(imgs.size(0)):
            if count >= n:
                break

            img_tensor = imgs[i:i+1]
            true_label = labels[i].item()

            img = imgs[i].cpu().permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            img_uint8 = (img * 255).astype(np.uint8)

            heatmap, pred_label, confidence = compute_gradcam(model, img_tensor)

            superimposed, _ = overlay_heatmap(img_uint8, heatmap)

            axes[count].imshow(superimposed)
            axes[count].axis("off")

            color = "green" if true_label == pred_label else "red"

            axes[count].set_title(
                f"T:{class_names[true_label]}\nP:{class_names[pred_label]} ({confidence:.2f})",
                fontsize=9,
                color=color
            )

            count += 1

        if count >= n:
            break

    for j in range(count, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Grad-CAM — Test Samples", fontsize=15, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_gradcam_grid(test_loader, model, class_names, n=8, cols=4)

In [ ]:
def plot_gradcam_wrong(test_loader, model, class_names, n=8, cols=4):
    wrong_imgs, wrong_true, wrong_pred, wrong_conf, wrong_heatmaps = [], [], [], [], []

    model.eval()

    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)

        for i in range(imgs.size(0)):
            img_tensor = imgs[i:i+1]
            true_label = labels[i].item()

            img = imgs[i].cpu().permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            img_u8 = (img * 255).astype(np.uint8)

            heatmap, pred_label, confidence = compute_gradcam(model, img_tensor)

            if pred_label != true_label:
                wrong_imgs.append(img_u8)
                wrong_true.append(true_label)
                wrong_pred.append(pred_label)
                wrong_conf.append(confidence)
                wrong_heatmaps.append(heatmap)

            if len(wrong_imgs) >= n:
                break

        if len(wrong_imgs) >= n:
            break

    rows = (len(wrong_imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    for idx, (img_u8, tl, pl, conf, hm) in enumerate(
            zip(wrong_imgs, wrong_true, wrong_pred, wrong_conf, wrong_heatmaps)):

        superimposed, _ = overlay_heatmap(img_u8, hm)

        axes[idx].imshow(superimposed)
        axes[idx].axis("off")

        axes[idx].set_title(
            f"T:{class_names[tl]}\nP:{class_names[pl]} ({conf:.2f})",
            fontsize=9,
            color="red"
        )

    for j in range(len(wrong_imgs), len(axes)):
        axes[j].axis("off")

    plt.suptitle("Grad-CAM — Wrong Predictions Only", fontsize=15, fontweight="bold", color="red")
    plt.tight_layout()
    plt.show()

    print(f"Total wrong shown: {len(wrong_imgs)}")

In [ ]:
plot_gradcam_wrong(test_loader, model, class_names, n=8, cols=4)

In [ ]:
def plot_gradcam_per_class(test_loader, model, class_names, cols=4):
    seen = {}

    model.eval()

    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)

        for i in range(imgs.size(0)):
            true_label = labels[i].item()

            if true_label in seen:
                continue

            img_tensor = imgs[i:i+1]

            img = imgs[i].cpu().permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            img_u8 = (img * 255).astype(np.uint8)

            heatmap, pred_label, confidence = compute_gradcam(model, img_tensor)

            if pred_label == true_label:
                seen[true_label] = (img_u8, heatmap, confidence)

        if len(seen) == len(class_names):
            break

    n = len(seen)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    for idx, cls_idx in enumerate(sorted(seen.keys())):
        img_u8, hm, conf = seen[cls_idx]

        superimposed, _ = overlay_heatmap(img_u8, hm)

        axes[idx].imshow(superimposed)
        axes[idx].axis("off")

        axes[idx].set_title(
            f"{class_names[cls_idx]}\n({conf:.2f})",
            fontsize=10,
            color="green"
        )

    for j in range(n, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Grad-CAM — One Correct Sample Per Class", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_gradcam_per_class(test_loader, model, class_names, cols=4)

In [ ]:
def plot_confidence_distribution(test_loader, model, class_names):
    all_true, all_pred, all_conf = [], [], []

    model.eval()

    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)

        for i in range(imgs.size(0)):
            img_tensor = imgs[i:i+1]

            img_tensor.requires_grad_(True)

            _, pred_label, confidence = compute_gradcam(model, img_tensor)

            all_true.append(labels[i].item())
            all_pred.append(pred_label)
            all_conf.append(confidence)

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    all_conf = np.array(all_conf)

    correct = all_conf[all_true == all_pred]
    wrong = all_conf[all_true != all_pred]

    plt.figure(figsize=(12, 5))
    plt.hist(correct, bins=20, alpha=0.7, color="green", label="Correct")
    plt.hist(wrong, bins=20, alpha=0.7, color="red", label="Wrong")

    plt.xlabel("Confidence")
    plt.ylabel("Count")
    plt.title("Confidence Distribution — Correct vs Wrong")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_confidence_distribution(test_loader, model, class_names)

In [ ]:
def plot_per_class_accuracy(test_loader, model, class_names):
    correct_count = {i: 0 for i in range(len(class_names))}
    total_count = {i: 0 for i in range(len(class_names))}

    model.eval()

    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)

        for i in range(imgs.size(0)):
            img_tensor = imgs[i:i+1]

            img_tensor.requires_grad = True

            _, pred_label, _ = compute_gradcam(model, img_tensor)

            true_label = labels[i].item()

            total_count[true_label] += 1
            if pred_label == true_label:
                correct_count[true_label] += 1

    accuracies = [
        correct_count[i] / total_count[i] if total_count[i] > 0 else 0
        for i in range(len(class_names))
    ]

    colors = [
        "green" if a >= 0.7 else "orange" if a >= 0.4 else "red"
        for a in accuracies
    ]

    plt.figure(figsize=(10, 5))

    bars = plt.bar(class_names, accuracies, color=colors, edgecolor="black")

    for bar, acc in zip(bars, accuracies):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{acc:.2f}",
            ha="center",
            fontsize=11,
            fontweight="bold"
        )

    plt.ylim(0, 1.1)
    plt.xlabel("Class")
    plt.ylabel("Accuracy")
    plt.title("Per-Class Accuracy (PyTorch)")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_per_class_accuracy(test_loader, model, class_names)

In [ ]:
def predict_and_gradcam(image):
    if image is None:
        return None, None, None, {}

    img = cv2.resize(image, (224, 224))
    img = img / 255.0

    img_uint8 = (img * 255).astype(np.uint8)

    img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    img_tensor.requires_grad_(True)

    heatmap, pred_label, confidence = compute_gradcam(model, img_tensor)

    superimposed, heatmap_rgb = overlay_heatmap(img_uint8, heatmap)

    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    probs_dict = {class_names[i]: float(probs[i]) for i in range(len(class_names))}

    return superimposed, heatmap_rgb, img_uint8, probs_dict


with gr.Blocks(title="ECG Grad-CAM Inference") as demo:
    gr.Markdown("# ECG Classification with Grad-CAM")
    gr.Markdown("Upload an ECG image to get prediction and Grad-CAM visualization.")

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="numpy", label="Upload ECG Image")
            run_btn = gr.Button("Run Inference", variant="primary")

        with gr.Column(scale=2):
            with gr.Row():
                out_original = gr.Image(label="Original")
                out_heatmap = gr.Image(label="Heatmap Only")
                out_overlay = gr.Image(label="Grad-CAM Overlay")

            out_probs = gr.Label(label="Class Probabilities")

    run_btn.click(
        fn=predict_and_gradcam,
        inputs=input_image,
        outputs=[out_overlay, out_heatmap, out_original, out_probs]
    )

demo.launch(share=True, debug=True)